Perfect 💡 — Let’s now upgrade your **LangGraph multi-agent framework** with
🧠 **memory integration**, so it can *remember previous conversations* and use that context intelligently across turns.

This gives you a **stateful AI agent system**, not just a stateless query handler.
If the user says “What about tomorrow?” after asking “What’s the weather in Delhi?”, the router and Weather MCP can understand it’s still about Delhi’s weather.

---

## 🚀 Goals of This Upgrade

✅ Add **conversational memory** (context persists across invocations).
✅ Router and fallback LLMs can access **previous user + system messages**.
✅ Still **fully dynamic** — auto-discovers MCP agents and routes to them.
✅ Uses **LangGraph’s built-in Memory integration** (with `MemorySaver`).

---

## 📂 Updated Structure

```
langgraph_project/
│
├── main.py
├── parent_graph.py
│
├── mcp_agents/
│   ├── __init__.py
│   ├── weather_agent.py
│   ├── math_agent.py
│   └── search_agent.py
│
└── .env
```

---

## 🧠 Updated `parent_graph.py`

Now includes **LangGraph Memory** and context-aware reasoning.

```python
# parent_graph.py

import os
import asyncio
import importlib
import pkgutil
from typing import Dict, Any, Callable
from dotenv import load_dotenv
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI

# -------------------------------
# Load environment variables
# -------------------------------
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("⚠️ Please set OPENAI_API_KEY in your .env file")

# -------------------------------
# Base LLM
# -------------------------------
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.4, api_key=OPENAI_API_KEY)

# -------------------------------
# Graph state
# -------------------------------
class GraphState(Dict[str, Any]):
    pass


# -------------------------------
# Dynamic MCP agent loader
# -------------------------------
def load_mcp_agents() -> Dict[str, Callable]:
    """Dynamically discover MCP agents (each defines run_agent())."""
    agents = {}
    package = "mcp_agents"

    for _, name, _ in pkgutil.iter_modules([package]):
        module = importlib.import_module(f"{package}.{name}")
        if hasattr(module, "run_agent"):
            key = name.replace("_agent", "")
            agents[key] = getattr(module, "run_agent")
            print(f"✅ Loaded MCP Agent: {key}")
    return agents


# -------------------------------
# Router Node (context-aware)
# -------------------------------
async def router_node(state: GraphState):
    """LLM router that chooses the best MCP agent or fallback."""
    user_input = state["user_input"]
    history = state.get("history", [])

    available_agents = load_mcp_agents()
    available_names = ", ".join(available_agents.keys())

    system_prompt = (
        "You are a routing controller. Decide which module should handle the query.\n"
        f"Available agents: {available_names}\n"
        "If none apply, choose 'fallback'. Respond only as JSON:\n"
        "{\"route\": \"<agent_name or fallback>\", \"reason\": \"<why you chose it>\"}\n\n"
        "Consider previous conversation for context if relevant."
    )

    messages = [{"role": "system", "content": system_prompt}]
    for msg in history[-5:]:  # include last few messages
        messages.append(msg)
    messages.append({"role": "user", "content": user_input})

    resp = await llm.ainvoke(messages)
    text = resp.content.strip()
    print(f"🔍 Router decision: {text}")

    route = "fallback"
    for name in available_agents.keys():
        if name.lower() in text.lower():
            route = name
            break

    return {**state, "route": route, "reason": text}


# -------------------------------
# Fallback Node (context-aware)
# -------------------------------
async def fallback_node(state: GraphState):
    """Fallback general LLM conversation handler."""
    user_input = state["user_input"]
    history = state.get("history", [])

    messages = [{"role": "system", "content": "You are a helpful conversational AI."}]
    messages.extend(history[-5:])
    messages.append({"role": "user", "content": user_input})

    response = await llm.ainvoke(messages)
    return {
        **state,
        "result": response.content.strip(),
        "handled_by": "fallback",
        "history": history + [{"role": "user", "content": user_input}, {"role": "assistant", "content": response.content}],
    }


# -------------------------------
# Build LangGraph with Memory
# -------------------------------
def build_langgraph():
    """Constructs the LangGraph with memory-enabled routing."""
    agents = load_mcp_agents()
    memory = MemorySaver()
    graph = StateGraph(GraphState)

    # Router node
    graph.add_node("router", router_node)
    graph.set_entry_point("router")

    # MCP nodes
    for name, func in agents.items():
        async def node(state: GraphState, _func=func, _name=name):
            result = _func(state["user_input"])
            history = state.get("history", [])
            history.append({"role": "user", "content": state["user_input"]})
            history.append({"role": "assistant", "content": result})
            return {**state, "result": result, "handled_by": _name, "history": history}

        graph.add_node(name, node)

    # Fallback node
    graph.add_node("fallback", fallback_node)

    # Conditional routing
    graph.add_conditional_edges(
        "router",
        lambda state: state["route"],
        {**{name: name for name in agents.keys()}, "fallback": "fallback"},
    )

    # Exit edges
    for node_name in [*agents.keys(), "fallback"]:
        graph.add_edge(node_name, END)

    # Compile with memory support
    return graph.compile(checkpointer=memory)
```

---

## 🧩 MCP Agent Example: `mcp_agents/weather_agent.py`

```python
# mcp_agents/weather_agent.py
import random

def run_agent(query: str) -> str:
    """Mock weather logic."""
    city = next((w for w in query.split() if w.istitle()), "Unknown City")
    temp = random.choice([28, 30, 32, 35])
    return f"The weather in {city} is {temp}°C and sunny."
```

(Same for `math_agent.py` and `search_agent.py` as before.)

---

## 🚀 `main.py`

```python
# main.py

import asyncio
from parent_graph import build_langgraph

async def main():
    app = build_langgraph()
    print("🤖 LangGraph Multi-Agent System with Memory (type 'exit' to quit)\n")

    # Single persistent thread for memory
    thread_id = "conversation-001"

    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in {"exit", "quit"}:
            print("👋 Goodbye!")
            break

        state = {"user_input": user_input}
        result = await app.ainvoke(state, config={"configurable": {"thread_id": thread_id}})

        print("\n🔹 Final Result:", result.get("result"))
        print("🔸 Routed To:", result.get("handled_by", result.get("route")))
        print("🔸 Reasoning:", result.get("reason"), "\n")

if __name__ == "__main__":
    asyncio.run(main())
```

---

## 🧪 Example Run (Now with Memory!)

```
🤖 LangGraph Multi-Agent System with Memory (type 'exit' to quit)

✅ Loaded MCP Agent: weather
✅ Loaded MCP Agent: math
✅ Loaded MCP Agent: search

You: What's the weather in Delhi?
🔍 Router decision: {"route": "weather", "reason": "Detected weather-related question."}
🔹 Final Result: The weather in Delhi is 30°C and sunny.
🔸 Routed To: weather
🔸 Reasoning: {"route": "weather", "reason": "Detected weather-related question."}

You: What about tomorrow?
🔍 Router decision: {"route": "weather", "reason": "Refers to previous weather question."}
🔹 Final Result: The weather in Delhi is 29°C and sunny.
🔸 Routed To: weather
🔸 Reasoning: {"route": "weather", "reason": "Refers to previous weather question."}

You: Tell me a joke.
🔍 Router decision: {"route": "fallback", "reason": "General conversation detected."}
🔹 Final Result: Why do programmers prefer dark mode? Because light attracts bugs!
🔸 Routed To: fallback
🔸 Reasoning: {"route": "fallback", "reason": "General conversation detected."}
```

---

## 🧩 How Memory Works

| Component                                                  | Purpose                                              |
| ---------------------------------------------------------- | ---------------------------------------------------- |
| `MemorySaver()`                                            | Persists context in-memory (per conversation thread) |
| `thread_id`                                                | Unique conversation ID                               |
| `history`                                                  | Stored in state and shared across nodes              |
| `router_node` & `fallback_node`                            | Include previous 5 messages for context              |
| `app.ainvoke(..., config={"configurable": {"thread_id"}})` | Tells LangGraph which memory thread to use           |

---

## 🔮 Next-Level Extensions

Once this is working, we can extend it to:

* 💾 Use **SQLite or Redis Memory** for persistence across restarts
* ⚙️ Add **LangGraph visualizer** to see node routing paths
* 🌐 Deploy via **FastAPI endpoint with streaming responses**
* 🧩 Add more agents (Finance, News, Travel) dynamically

---

Would you like me to extend this next with **persistent memory using SQLite (so the context survives restarts)** and a **FastAPI endpoint** for your LangGraph multi-agent system?
